In [1]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
df = pd.read_csv('dataset_gojek.csv')
print(df.shape)
print(df['sentiment'].value_counts())
df.head()

(10000, 4)
sentiment
positif    7039
negatif    2584
netral      377
Name: count, dtype: int64


,username,review,rating,sentiment
0,Pengguna Google,sangat membantu,5,positif
1,Pengguna Google,"Sangat membantu,cepat dan ramah driver nya",5,positif
2,Pengguna Google,"saldo multi trip saya ga masuk masuk, keterang...",1,negatif
3,Pengguna Google,👍👍👍👍,5,positif
4,Pengguna Google,"okee banget aplikasinya, mantulll tapi sayang ...",5,positif


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['review'].apply(clean_text)
df = df[df['review_clean'].str.strip() != '']
df = df.dropna(subset=['review_clean'])

print(f"Total data setelah cleaning: {len(df)}")
df[['review', 'review_clean', 'sentiment']].head()

Total data setelah cleaning: 9897


,review,review_clean,sentiment
0,sangat membantu,sangat membantu,positif
1,"Sangat membantu,cepat dan ramah driver nya",sangat membantucepat dan ramah driver nya,positif
2,"saldo multi trip saya ga masuk masuk, keterang...",saldo multi trip saya ga masuk masuk keteranga...,negatif
4,"okee banget aplikasinya, mantulll tapi sayang ...",okee banget aplikasinya mantulll tapi sayang m...,positif
5,top,top,positif


In [5]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])
print("Mapping label:", dict(zip(le.classes_, le.transform(le.classes_))))
print(df['label'].value_counts())

Mapping label: {'negatif': np.int64(0), 'netral': np.int64(1), 'positif': np.int64(2)}
label
2    6940
0    2581
1     376
Name: count, dtype: int64


In [6]:
MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['review_clean'])

X = pad_sequences(tokenizer.texts_to_sequences(df['review_clean']), maxlen=MAX_LEN)
y = df['label'].values

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Shape X: (9897, 100)
Shape y: (9897,)


In [7]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model1 = Sequential([
    Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model1.summary()

history1 = model1.fit(
    X_train1, y_train1,
    epochs=10,
    batch_size=64,
    validation_data=(X_test1, y_test1),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

y_pred1 = np.argmax(model1.predict(X_test1), axis=1)
print("\n=== Skema 1: LSTM 80/20 ===")
print(f"Akurasi: {accuracy_score(y_test1, y_pred1):.4f}")
print(classification_report(y_test1, y_pred1, target_names=le.classes_))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 22s 142ms/step - accuracy: 0.7902 - loss: 0.5482 - val_accuracy: 0.8566 - val_loss: 0.4104
Epoch 2/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 19s 135ms/step - accuracy: 0.8798 - loss: 0.3483 - val_accuracy: 0.8843 - val_loss: 0.3387
Epoch 3/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 21s 137ms/step - accuracy: 0.9169 - loss: 0.2561 - val_accuracy: 0.8899 - val_loss: 0.3381
Epoch 4/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 17s 136ms/step - accuracy: 0.9400 - loss: 0.1957 - val_accuracy: 0.8838 - val_loss: 0.3576
Epoch 5/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 18s 148ms/step - accuracy: 0.9587 - loss: 0.1414 - val_accuracy: 0.8843 - val_loss: 0.4038
Epoch 6/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 19s 138ms/step - accuracy: 0.9658 - loss: 0.1186 - val_accuracy: 0.8657 - val_loss: 0.5097
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step

=== Skema 1: LSTM 80/20 ===
Akurasi: 0.8899
              precision    recall  f1-score   support

     negatif       0.77      0.87      0.82       516
      netral       0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

model2 = Sequential([
    Embedding(MAX_WORDS, 64),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history2 = model2.fit(
    X_train2, y_train2,
    epochs=10,
    batch_size=64,
    validation_data=(X_test2, y_test2),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

y_pred2 = np.argmax(model2.predict(X_test2), axis=1)
print("\n=== Skema 2: BiLSTM 80/20 ===")
print(f"Akurasi: {accuracy_score(y_test2, y_pred2):.4f}")
print(classification_report(y_test2, y_pred2, target_names=le.classes_))

Epoch 1/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 42s 285ms/step - accuracy: 0.8102 - loss: 0.5191 - val_accuracy: 0.8778 - val_loss: 0.3470
Epoch 2/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 34s 274ms/step - accuracy: 0.8991 - loss: 0.3122 - val_accuracy: 0.8753 - val_loss: 0.3417
Epoch 3/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 44s 298ms/step - accuracy: 0.9251 - loss: 0.2476 - val_accuracy: 0.8823 - val_loss: 0.3405
Epoch 4/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 39s 286ms/step - accuracy: 0.9367 - loss: 0.2090 - val_accuracy: 0.8606 - val_loss: 0.4114
Epoch 5/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 41s 283ms/step - accuracy: 0.9454 - loss: 0.1823 - val_accuracy: 0.8611 - val_loss: 0.4376
Epoch 6/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 40s 279ms/step - accuracy: 0.9523 - loss: 0.1570 - val_accuracy: 0.8556 - val_loss: 0.4696
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step

=== Skema 2: BiLSTM 80/20 ===
Akurasi: 0.8823
              precision    recall  f1-score   support

     negatif       0.74      0.89      0.81       516
      netral      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model3 = Sequential([
    Embedding(MAX_WORDS, 64),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model3.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history3 = model3.fit(
    X_train3, y_train3,
    epochs=10,
    batch_size=64,
    validation_data=(X_test3, y_test3),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

y_pred3 = np.argmax(model3.predict(X_test3), axis=1)
print("\n=== Skema 3: LSTM 70/30 ===")
print(f"Akurasi: {accuracy_score(y_test3, y_pred3):.4f}")
print(classification_report(y_test3, y_pred3, target_names=le.classes_))

Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 21s 157ms/step - accuracy: 0.7902 - loss: 0.5591 - val_accuracy: 0.8404 - val_loss: 0.4413
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 153ms/step - accuracy: 0.8777 - loss: 0.3549 - val_accuracy: 0.8811 - val_loss: 0.3397
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 19s 144ms/step - accuracy: 0.9160 - loss: 0.2526 - val_accuracy: 0.8761 - val_loss: 0.3801
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 144ms/step - accuracy: 0.9456 - loss: 0.1852 - val_accuracy: 0.8710 - val_loss: 0.3864
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 21s 153ms/step - accuracy: 0.9606 - loss: 0.1324 - val_accuracy: 0.8606 - val_loss: 0.4449
93/93 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step

=== Skema 3: LSTM 70/30 ===
Akurasi: 0.8811
              precision    recall  f1-score   support

     negatif       0.75      0.86      0.81       774
      netral       0.00      0.00      0.00       113
     positif       0.94      0.94      0.94      2083

    accuracy                           0.8

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# Inference - test model dengan kalimat baru
def predict_sentiment(text, model):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN)
    pred = np.argmax(model.predict(padded), axis=1)
    return le.inverse_transform(pred)[0]

# Test beberapa kalimat
test_texts = [
    "aplikasi gojek sangat membantu dan mudah digunakan",
    "driver nya tidak sopan dan sering cancel orderan",
    "biasa aja sih aplikasinya, tidak ada yang spesial"
]

print("=== Hasil Inference ===")
for text in test_texts:
    result = predict_sentiment(text, model1)
    print(f"Teks: '{text}'")
    print(f"Sentimen: {result}\n")

=== Hasil Inference ===
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Teks: 'aplikasi gojek sangat membantu dan mudah digunakan'
Sentimen: positif

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Teks: 'driver nya tidak sopan dan sering cancel orderan'
Sentimen: positif

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Teks: 'biasa aja sih aplikasinya, tidak ada yang spesial'
Sentimen: negatif



In [11]:
print("=" * 50)
print("RINGKASAN PERBANDINGAN SKEMA PELATIHAN")
print("=" * 50)
print(f"Skema 1 - LSTM, Split 80/20     : {accuracy_score(y_test1, y_pred1):.4f}")
print(f"Skema 2 - BiLSTM, Split 80/20   : {accuracy_score(y_test2, y_pred2):.4f}")
print(f"Skema 3 - LSTM, Split 70/30     : {accuracy_score(y_test3, y_pred3):.4f}")
print("=" * 50)

RINGKASAN PERBANDINGAN SKEMA PELATIHAN
Skema 1 - LSTM, Split 80/20     : 0.8899
Skema 2 - BiLSTM, Split 80/20   : 0.8823
Skema 3 - LSTM, Split 70/30     : 0.8811


In [12]:
# Buat file requirements.txt
requirements = """google-play-scraper==1.2.7
tensorflow
scikit-learn
pandas
numpy
matplotlib
seaborn
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt berhasil dibuat!")

requirements.txt berhasil dibuat!
